# M5 Phase 1 + 2: Hiểu dữ liệu gốc và join để tái tạo doanh thu

Notebook này chỉ thực hiện Phase 1 và Phase 2 cho bộ dữ liệu M5:

- đọc và kiểm tra dữ liệu gốc;
- so sánh hai file sales train;
- hiểu schema, cấp phân cấp sản phẩm/cửa hàng;
- nối sales với calendar và sell prices;
- tái tạo doanh thu ở mức item-day khi chất lượng join cho phép;
- tạo hai bảng ứng viên ở mức `store-dept-day` và `store-day`.

Không huấn luyện mô hình trong notebook này. Các cột phát sinh từ doanh số cùng ngày sẽ được giữ cho EDA nhưng được đánh dấu là có nguy cơ leakage nếu dùng để dự báo doanh thu trước khi ngày đó xảy ra.

## 1. Setup và kiểm tra file

Cell dưới đây tìm project root theo thư mục dữ liệu M5, khai báo đường dẫn và kiểm tra các file bắt buộc. Việc kiểm tra này giúp tránh giả định rằng dữ liệu đã đầy đủ trước khi đọc.

In [1]:
from pathlib import Path
import gc
import math
import os
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    if "__file__" in globals():
        file_parent = Path(__file__).resolve().parent
        candidates = [file_parent, *file_parent.parents, *candidates]
    for candidate in candidates:
        if (candidate / "data" / "m5-forecasting-accuracy").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục data/m5-forecasting-accuracy từ vị trí hiện tại.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "m5-forecasting-accuracy"
EXPECTED_FILES = [
    "calendar.csv",
    "sell_prices.csv",
    "sales_train_evaluation.csv",
    "sales_train_validation.csv",
    "sample_submission.csv",
]


def file_size_mb(path):
    return path.stat().st_size / (1024 ** 2) if path.exists() else np.nan


file_inventory = pd.DataFrame(
    [
        {
            "file": name,
            "exists": (DATA_DIR / name).exists(),
            "size_mb": file_size_mb(DATA_DIR / name),
        }
        for name in EXPECTED_FILES
    ]
)

display(file_inventory)
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset folder: {DATA_DIR}")

missing_files = file_inventory.loc[~file_inventory["exists"], "file"].tolist()
if missing_files:
    print("Diễn giải: Thiếu file bắt buộc, chưa nên tiếp tục phân tích:", missing_files)
else:
    print("Diễn giải: Tất cả file M5 bắt buộc đều tồn tại, thư mục dữ liệu đủ điều kiện để bắt đầu Phase 1 + 2.")

,file,exists,size_mb
0,calendar.csv,True,0.0987
1,sell_prices.csv,True,193.9733
2,sales_train_evaluation.csv,True,116.0970
3,sales_train_validation.csv,True,114.4483
4,sample_submission.csv,True,4.9866


Project root: /Users/tnhatnguyendev2805/Documents/University/Data Science/Final Project
Dataset folder: /Users/tnhatnguyendev2805/Documents/University/Data Science/Final Project/data/m5-forecasting-accuracy
Diễn giải: Tất cả file M5 bắt buộc đều tồn tại, thư mục dữ liệu đủ điều kiện để bắt đầu Phase 1 + 2.


## 2. Đọc dữ liệu gốc và kiểm tra từng file

M5 có hai bảng sales rất rộng. Để notebook chạy ổn định hơn, các cột ngày `d_*` được đọc bằng kiểu `int16`, còn các biến phân loại được đọc bằng `category` hoặc `string`. Đây chỉ là tối ưu bộ nhớ, không thay đổi ý nghĩa dữ liệu.

In [2]:
def header_columns(csv_path):
    return pd.read_csv(csv_path, nrows=0).columns.tolist()


def split_sales_columns(columns):
    day_cols = [c for c in columns if c.startswith("d_")]
    meta_cols = [c for c in columns if not c.startswith("d_")]
    return meta_cols, day_cols


validation_header = header_columns(DATA_DIR / "sales_train_validation.csv")
evaluation_header = header_columns(DATA_DIR / "sales_train_evaluation.csv")
validation_meta_cols, validation_day_cols = split_sales_columns(validation_header)
evaluation_meta_cols, evaluation_day_cols = split_sales_columns(evaluation_header)


def sales_dtype(day_cols):
    dtype = {c: "int16" for c in day_cols}
    dtype.update(
        {
            "id": "string",
            "item_id": "string",
            "dept_id": "category",
            "cat_id": "category",
            "store_id": "category",
            "state_id": "category",
        }
    )
    return dtype


calendar = pd.read_csv(
    DATA_DIR / "calendar.csv",
    parse_dates=["date"],
    dtype={
        "wm_yr_wk": "int16",
        "weekday": "category",
        "wday": "int8",
        "month": "int8",
        "year": "int16",
        "d": "string",
        "event_name_1": "string",
        "event_type_1": "string",
        "event_name_2": "string",
        "event_type_2": "string",
        "snap_CA": "int8",
        "snap_TX": "int8",
        "snap_WI": "int8",
    },
)
sell_prices = pd.read_csv(
    DATA_DIR / "sell_prices.csv",
    dtype={
        "store_id": "category",
        "item_id": "string",
        "wm_yr_wk": "int16",
        "sell_price": "float32",
    },
)
sales_train_validation = pd.read_csv(
    DATA_DIR / "sales_train_validation.csv",
    dtype=sales_dtype(validation_day_cols),
)
sales_train_evaluation = pd.read_csv(
    DATA_DIR / "sales_train_evaluation.csv",
    dtype=sales_dtype(evaluation_day_cols),
)
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"id": "string"})

raw_tables = {
    "calendar": calendar,
    "sell_prices": sell_prices,
    "sales_train_validation": sales_train_validation,
    "sales_train_evaluation": sales_train_evaluation,
    "sample_submission": sample_submission,
}

shape_summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "memory_mb": df.memory_usage(deep=True).sum() / (1024 ** 2),
        }
        for name, df in raw_tables.items()
    ]
)
display(shape_summary)
print("Diễn giải: Dữ liệu đã được đọc thành công. Hai bảng sales là bảng rộng theo ngày, còn sell_prices là bảng lớn nhất theo số dòng.")

,table,rows,columns,memory_mb
0,calendar,1969,14,0.5584
1,sell_prices,6841121,4,448.6328
2,sales_train_validation,30490,1919,115.4250
3,sales_train_evaluation,30490,1947,117.0534
4,sample_submission,60980,29,17.5481


Diễn giải: Dữ liệu đã được đọc thành công. Hai bảng sales là bảng rộng theo ngày, còn sell_prices là bảng lớn nhất theo số dòng.


In [3]:
def column_preview(df, n_first=10, n_last=8):
    cols = list(df.columns)
    if len(cols) <= n_first + n_last:
        preview = cols
    else:
        preview = cols[:n_first] + ["..."] + cols[-n_last:]
    return pd.DataFrame({"column_preview": preview})


def first_rows_preview(df, rows=3):
    if df.shape[1] <= 16:
        return df.head(rows)
    keep_cols = list(df.columns[:8]) + list(df.columns[-6:])
    return df.loc[:, keep_cols].head(rows)


def missing_summary(df, top_n=20):
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    if miss.empty:
        return pd.DataFrame({"message": ["Không có missing value ở cấp cột."]})
    out = miss.head(top_n).rename("missing_count").to_frame()
    out["missing_rate"] = out["missing_count"] / len(df)
    out.index.name = "column"
    return out.reset_index()


def dtype_summary(df):
    return (
        pd.Series(df.dtypes.astype(str), name="dtype")
        .value_counts()
        .rename_axis("dtype")
        .reset_index(name="column_count")
    )


duplicate_checks = {
    "calendar": ["d"],
    "sell_prices": ["store_id", "item_id", "wm_yr_wk"],
    "sales_train_validation": ["id"],
    "sales_train_evaluation": ["id"],
    "sample_submission": ["id"],
}

inspection_rows = []
for name, df in raw_tables.items():
    print("\n" + "=" * 90)
    print(f"Bảng: {name}")
    print(f"Shape: {df.shape}")
    print("Cột:")
    display(column_preview(df))
    print("Một vài dòng đầu:")
    display(first_rows_preview(df))
    print("Tóm tắt dtype:")
    display(dtype_summary(df))
    print("Missing values:")
    display(missing_summary(df))
    key_cols = duplicate_checks[name]
    duplicate_key_count = int(df.duplicated(key_cols).sum())
    full_duplicate_count = int(df.duplicated().sum()) if len(df) <= 100_000 else np.nan
    inspection_rows.append(
        {
            "table": name,
            "row_unit_guess_after_inspection": {
                "calendar": "một ngày lịch M5",
                "sell_prices": "một giá bán theo store-item-week",
                "sales_train_validation": "một chuỗi item-store đến d_1913",
                "sales_train_evaluation": "một chuỗi item-store đến d_1941",
                "sample_submission": "một chuỗi dự báo 28 ngày theo id submission",
            }[name],
            "key_checked": " + ".join(key_cols),
            "duplicate_key_count": duplicate_key_count,
            "full_duplicate_count_if_checked": full_duplicate_count,
            "column_count_with_missing": int((df.isna().sum() > 0).sum()),
        }
    )

inspection_summary = pd.DataFrame(inspection_rows)
display(inspection_summary)
print(
    "Diễn giải: calendar và sell_prices đóng vai trò dimension/lookup theo ngày và tuần; "
    "hai bảng sales là fact table dạng wide theo item-store; sample_submission chủ yếu dùng để hiểu format nộp dự báo."
)


Bảng: calendar
Shape: (1969, 14)
Cột:


,column_preview
0,date
1,wm_yr_wk
2,weekday
3,wday
4,month
5,year
6,d
7,event_name_1
8,event_type_1
9,event_name_2


Một vài dòng đầu:


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,<NA>,<NA>,<NA>,<NA>,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,<NA>,<NA>,<NA>,<NA>,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,<NA>,<NA>,<NA>,<NA>,0,0,0


Tóm tắt dtype:


...

,dtype,column_count
0,int64,28
1,string,1


Missing values:


,message
0,Không có missing value ở cấp cột.


,table,row_unit_guess_after_inspection,key_checked,duplicate_key_count,full_duplicate_count_if_checked,column_count_with_missing
0,calendar,một ngày lịch M5,d,0,0.0000,4
1,sell_prices,một giá bán theo store-item-week,store_id + item_id + wm_yr_wk,0,NaN,0
2,sales_train_validation,một chuỗi item-store đến d_1913,id,0,0.0000,0
3,sales_train_evaluation,một chuỗi item-store đến d_1941,id,0,0.0000,0
4,sample_submission,một chuỗi dự báo 28 ngày theo id submission,id,0,0.0000,0


Diễn giải: calendar và sell_prices đóng vai trò dimension/lookup theo ngày và tuần; hai bảng sales là fact table dạng wide theo item-store; sample_submission chủ yếu dùng để hiểu format nộp dự báo.


## 3. So sánh `sales_train_validation.csv` và `sales_train_evaluation.csv`

Hai file sales có cấu trúc gần giống nhau nhưng có thể khác số ngày. Không chọn file chính trước khi kiểm tra số cột ngày và metadata.

In [4]:
common_metadata_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
validation_id_base = sales_train_validation["id"].str.replace("_validation$", "", regex=True)
evaluation_id_base = sales_train_evaluation["id"].str.replace("_evaluation$", "", regex=True)

sales_compare = pd.DataFrame(
    [
        {
            "file": "sales_train_validation.csv",
            "rows": sales_train_validation.shape[0],
            "columns": sales_train_validation.shape[1],
            "day_column_count": len(validation_day_cols),
            "first_day_column": validation_day_cols[0],
            "last_day_column": validation_day_cols[-1],
        },
        {
            "file": "sales_train_evaluation.csv",
            "rows": sales_train_evaluation.shape[0],
            "columns": sales_train_evaluation.shape[1],
            "day_column_count": len(evaluation_day_cols),
            "first_day_column": evaluation_day_cols[0],
            "last_day_column": evaluation_day_cols[-1],
        },
    ]
)
metadata_same = sales_train_validation[common_metadata_cols].equals(sales_train_evaluation[common_metadata_cols])
base_id_same = validation_id_base.equals(evaluation_id_base)
extra_eval_days = len(evaluation_day_cols) - len(validation_day_cols)

display(sales_compare)
print(f"Metadata item/dept/category/store/state giống nhau giữa hai file: {metadata_same}")
print(f"ID giống nhau nếu bỏ hậu tố _validation/_evaluation: {base_id_same}")
print(f"Số cột ngày evaluation nhiều hơn validation: {extra_eval_days}")

if metadata_same and base_id_same and len(evaluation_day_cols) >= len(validation_day_cols):
    selected_sales_name = "sales_train_evaluation.csv"
    selected_sales = sales_train_evaluation
    selected_day_cols = evaluation_day_cols
    selected_meta_cols = evaluation_meta_cols
    print(
        "Quyết định: dùng sales_train_evaluation.csv vì có cùng chuỗi item-store như validation "
        "nhưng bao phủ nhiều ngày hơn, nên phù hợp hơn để tái tạo doanh thu lịch sử."
    )
else:
    selected_sales_name = "sales_train_validation.csv"
    selected_sales = sales_train_validation
    selected_day_cols = validation_day_cols
    selected_meta_cols = validation_meta_cols
    print(
        "Quyết định: dùng sales_train_validation.csv vì so sánh metadata/date cho thấy evaluation không an toàn hơn trong lần kiểm tra này."
    )

,file,rows,columns,day_column_count,first_day_column,last_day_column
0,sales_train_validation.csv,30490,1919,1913,d_1,d_1913
1,sales_train_evaluation.csv,30490,1947,1941,d_1,d_1941


Metadata item/dept/category/store/state giống nhau giữa hai file: True
ID giống nhau nếu bỏ hậu tố _validation/_evaluation: True
Số cột ngày evaluation nhiều hơn validation: 28
Quyết định: dùng sales_train_evaluation.csv vì có cùng chuỗi item-store như validation nhưng bao phủ nhiều ngày hơn, nên phù hợp hơn để tái tạo doanh thu lịch sử.


## 4. Hiểu schema và phân cấp sản phẩm/cửa hàng

Phần này kiểm tra các cấp phân cấp thật sự xuất hiện trong file sales đã chọn: state, store, category, department, item và item-store series.

In [5]:
hierarchy_summary = pd.DataFrame(
    [
        {"metric": "states", "value": selected_sales["state_id"].nunique()},
        {"metric": "stores", "value": selected_sales["store_id"].nunique()},
        {"metric": "categories", "value": selected_sales["cat_id"].nunique()},
        {"metric": "departments", "value": selected_sales["dept_id"].nunique()},
        {"metric": "items", "value": selected_sales["item_id"].nunique()},
        {"metric": "item-store series", "value": selected_sales[["item_id", "store_id"]].drop_duplicates().shape[0]},
    ]
)
store_by_state = (
    selected_sales[["state_id", "store_id"]]
    .drop_duplicates()
    .groupby("state_id", observed=True)["store_id"]
    .nunique()
    .reset_index(name="store_count")
)
dept_by_category = (
    selected_sales[["cat_id", "dept_id"]]
    .drop_duplicates()
    .sort_values(["cat_id", "dept_id"])
    .groupby("cat_id", observed=True)
    .agg(dept_count=("dept_id", "nunique"), departments=("dept_id", lambda s: ", ".join(map(str, s))))
    .reset_index()
)
item_by_dept = (
    selected_sales[["cat_id", "dept_id", "item_id"]]
    .drop_duplicates()
    .groupby(["cat_id", "dept_id"], observed=True)
    .size()
    .reset_index(name="item_count")
    .sort_values(["cat_id", "dept_id"])
)

display(hierarchy_summary)
display(store_by_state)
display(dept_by_category)
display(item_by_dept)
print(
    "Diễn giải: grain tự nhiên của sales gốc là một chuỗi item-store, còn các cột d_* là số units theo ngày. "
    "Phân tích theo dept hoặc cat có ý nghĩa vì chúng là cấp sản phẩm ổn định hơn item riêng lẻ."
)

,metric,value
0,states,3
1,stores,10
2,categories,3
3,departments,7
4,items,3049
5,item-store series,30490


,state_id,store_count
0,CA,4
1,TX,3
2,WI,3


,cat_id,dept_count,departments
0,HOBBIES,2,"HOBBIES_1, HOBBIES_2"
1,HOUSEHOLD,2,"HOUSEHOLD_1, HOUSEHOLD_2"
2,FOODS,3,"FOODS_1, FOODS_2, FOODS_3"


,cat_id,dept_id,item_count
0,HOBBIES,HOBBIES_1,416
1,HOBBIES,HOBBIES_2,149
2,HOUSEHOLD,HOUSEHOLD_1,532
3,HOUSEHOLD,HOUSEHOLD_2,515
4,FOODS,FOODS_1,216
5,FOODS,FOODS_2,398
6,FOODS,FOODS_3,823


Diễn giải: grain tự nhiên của sales gốc là một chuỗi item-store, còn các cột d_* là số units theo ngày. Phân tích theo dept hoặc cat có ý nghĩa vì chúng là cấp sản phẩm ổn định hơn item riêng lẻ.


## 5. Chuẩn bị calendar và kiểm tra liên kết với cột `d`

Sales wide dùng các cột `d_1`, `d_2`, ... nên cần kiểm tra trực tiếp rằng các mã `d` này có trong `calendar.csv`. Sau đó mới tạo feature thời gian.

In [6]:
calendar_prepared = calendar.copy()
iso_calendar = calendar_prepared["date"].dt.isocalendar()
calendar_prepared["quarter"] = calendar_prepared["date"].dt.quarter.astype("int8")
calendar_prepared["week_of_year"] = iso_calendar.week.astype("int16")
calendar_prepared["day_of_month"] = calendar_prepared["date"].dt.day.astype("int8")
calendar_prepared["day_of_year"] = calendar_prepared["date"].dt.dayofyear.astype("int16")
calendar_prepared["day_of_week"] = calendar_prepared["date"].dt.day_name()
calendar_prepared["day_of_week_num"] = calendar_prepared["date"].dt.dayofweek.astype("int8")
calendar_prepared["is_weekend"] = calendar_prepared["day_of_week_num"].isin([5, 6]).astype("int8")
calendar_prepared["year_month"] = calendar_prepared["date"].dt.to_period("M").astype(str)
calendar_prepared["has_event_1"] = calendar_prepared["event_name_1"].notna().astype("int8")
calendar_prepared["has_event_2"] = calendar_prepared["event_name_2"].notna().astype("int8")
calendar_prepared["event_count"] = (
    calendar_prepared[["has_event_1", "has_event_2"]].sum(axis=1).astype("int8")
)

selected_day_set = set(selected_day_cols)
calendar_day_set = set(calendar_prepared["d"].astype(str))
missing_sales_days_in_calendar = sorted(selected_day_set - calendar_day_set)
calendar_relevant = (
    calendar_prepared[calendar_prepared["d"].isin(selected_day_cols)]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

calendar_join_cols = [
    "d",
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "quarter",
    "week_of_year",
    "day_of_month",
    "day_of_year",
    "day_of_week",
    "day_of_week_num",
    "is_weekend",
    "year_month",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "event_count",
    "snap_CA",
    "snap_TX",
    "snap_WI",
]
calendar_join = calendar_relevant[calendar_join_cols].copy()

calendar_summary = pd.DataFrame(
    [
        {"metric": "selected_sales_day_columns", "value": len(selected_day_cols)},
        {"metric": "calendar_rows_relevant_to_selected_sales", "value": len(calendar_relevant)},
        {"metric": "missing_sales_days_in_calendar", "value": len(missing_sales_days_in_calendar)},
        {"metric": "date_min", "value": calendar_relevant["date"].min()},
        {"metric": "date_max", "value": calendar_relevant["date"].max()},
        {"metric": "wm_yr_wk_count", "value": calendar_relevant["wm_yr_wk"].nunique()},
    ]
)
event_missing = calendar_relevant[["event_name_1", "event_type_1", "event_name_2", "event_type_2"]].isna().sum().reset_index()
event_missing.columns = ["event_column", "missing_count"]
event_missing["missing_rate"] = event_missing["missing_count"] / len(calendar_relevant)
snap_summary = calendar_relevant[["snap_CA", "snap_TX", "snap_WI"]].agg(["sum", "mean"]).T.reset_index()
snap_summary.columns = ["snap_column", "days_active", "active_rate"]

display(calendar_summary)
display(event_missing)
display(snap_summary)
display(calendar_join.head())
print(
    "Diễn giải: calendar nối với sales bằng cột d. Các cột event bị thiếu chủ yếu biểu thị ngày không có sự kiện, "
    "không nên tự xem là lỗi dữ liệu. SNAP là cờ theo từng bang, nên khi aggregate theo store cần chọn SNAP tương ứng với state_id của store."
)

,metric,value
0,selected_sales_day_columns,1941
1,calendar_rows_relevant_to_selected_sales,1941
2,missing_sales_days_in_calendar,0
3,date_min,2011-01-29 00:00:00
4,date_max,2016-05-22 00:00:00
5,wm_yr_wk_count,278


,event_column,missing_count,missing_rate
0,event_name_1,1783,0.9186
1,event_type_1,1783,0.9186
2,event_name_2,1937,0.9979
3,event_type_2,1937,0.9979


,snap_column,days_active,active_rate
0,snap_CA,640.0000,0.3297
1,snap_TX,640.0000,0.3297
2,snap_WI,640.0000,0.3297


,d,date,wm_yr_wk,weekday,wday,month,year,quarter,week_of_year,day_of_month,day_of_year,day_of_week,day_of_week_num,is_weekend,year_month,event_name_1,event_type_1,event_name_2,event_type_2,event_count,snap_CA,snap_TX,snap_WI
0,d_1,2011-01-29,11101,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0
1,d_2,2011-01-30,11101,Sunday,2,1,2011,1,4,30,30,Sunday,6,1,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0
2,d_3,2011-01-31,11101,Monday,3,1,2011,1,5,31,31,Monday,0,0,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0
3,d_4,2011-02-01,11101,Tuesday,4,2,2011,1,5,1,32,Tuesday,1,0,2011-02,<NA>,<NA>,<NA>,<NA>,0,1,1,0
4,d_5,2011-02-02,11101,Wednesday,5,2,2011,1,5,2,33,Wednesday,2,0,2011-02,<NA>,<NA>,<NA>,<NA>,0,1,0,1


Diễn giải: calendar nối với sales bằng cột d. Các cột event bị thiếu chủ yếu biểu thị ngày không có sự kiện, không nên tự xem là lỗi dữ liệu. SNAP là cờ theo từng bang, nên khi aggregate theo store cần chọn SNAP tương ứng với state_id của store.


## 6. Kiểm tra bảng giá `sell_prices.csv`

Giá trong M5 không nằm ở mức transaction hay ngày, mà ở mức `store_id + item_id + wm_yr_wk`. Vì vậy doanh thu ngày chỉ có thể được tái tạo bằng units theo ngày nhân với giá tuần tương ứng.

In [7]:
price_key_cols = ["store_id", "item_id", "wm_yr_wk"]
price_duplicate_keys = int(sell_prices.duplicated(price_key_cols).sum())
price_missing = sell_prices.isna().sum().rename("missing_count").reset_index().rename(columns={"index": "column"})
price_weeks = set(sell_prices["wm_yr_wk"].unique())
sales_weeks = set(calendar_relevant["wm_yr_wk"].unique())
missing_price_weeks_for_sales_calendar = sorted(sales_weeks - price_weeks)
price_weeks_outside_sales_calendar = sorted(price_weeks - sales_weeks)

price_summary = pd.DataFrame(
    [
        {"metric": "rows", "value": len(sell_prices)},
        {"metric": "unique_store_item_week_keys", "value": sell_prices[price_key_cols].drop_duplicates().shape[0]},
        {"metric": "duplicate_store_item_week_keys", "value": price_duplicate_keys},
        {"metric": "unique_stores", "value": sell_prices["store_id"].nunique()},
        {"metric": "unique_items", "value": sell_prices["item_id"].nunique()},
        {"metric": "unique_weeks", "value": sell_prices["wm_yr_wk"].nunique()},
        {"metric": "nonpositive_price_rows", "value": int((sell_prices["sell_price"] <= 0).sum())},
        {"metric": "sales_calendar_weeks_without_any_price_week", "value": len(missing_price_weeks_for_sales_calendar)},
        {"metric": "price_weeks_outside_selected_sales_calendar", "value": len(price_weeks_outside_sales_calendar)},
    ]
)
display(price_summary)
display(price_missing)
display(sell_prices["sell_price"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame("sell_price"))
print("Giá nhỏ nhất:")
display(sell_prices.nsmallest(10, "sell_price"))
print("Giá lớn nhất:")
display(sell_prices.nlargest(10, "sell_price"))
print(
    "Diễn giải: nếu key store-item-week là duy nhất và không có giá không dương, join giá sẽ không làm nổ dòng. "
    "Tuy vậy vẫn cần kiểm tra item-day có sales dương nhưng thiếu price sau khi nối."
)

,metric,value
0,rows,6841121
1,unique_store_item_week_keys,6841121
2,duplicate_store_item_week_keys,0
3,unique_stores,10
4,unique_items,3049
5,unique_weeks,282
6,nonpositive_price_rows,0
7,sales_calendar_weeks_without_any_price_week,0
8,price_weeks_outside_selected_sales_calendar,4


,column,missing_count
0,store_id,0
1,item_id,0
2,wm_yr_wk,0
3,sell_price,0


,sell_price
count,"6,841,121.0000"
mean,4.4110
std,3.4088
min,0.0100
1%,0.4800
5%,0.9700
50%,3.4700
95%,10.9800
99%,17.9200
max,107.3200


Giá nhỏ nhất:


,store_id,item_id,wm_yr_wk,sell_price
225727,CA_1,HOUSEHOLD_1_443,11442,0.0100
1394085,CA_3,HOBBIES_1_261,11332,0.0100
1394086,CA_3,HOBBIES_1_261,11333,0.0100
1394087,CA_3,HOBBIES_1_261,11334,0.0100
2617306,CA_4,FOODS_3_413,11148,0.0100
4877375,WI_1,HOBBIES_1_338,11342,0.0100
4877376,WI_1,HOBBIES_1_338,11343,0.0100
5044267,WI_1,HOUSEHOLD_1_533,11408,0.0100
5316537,WI_1,FOODS_3_122,11425,0.0100
6282347,WI_3,HOUSEHOLD_1_036,11442,0.0100


Giá lớn nhất:


,store_id,item_id,wm_yr_wk,sell_price
6485945,WI_3,HOUSEHOLD_2_406,11317,107.3200
6485946,WI_3,HOUSEHOLD_2_406,11318,107.3200
6485947,WI_3,HOUSEHOLD_2_406,11319,107.3200
5805271,WI_2,HOUSEHOLD_2_406,11237,61.4600
5805272,WI_2,HOUSEHOLD_2_406,11238,61.4600
5805273,WI_2,HOUSEHOLD_2_406,11239,61.4600
5805274,WI_2,HOUSEHOLD_2_406,11240,61.4600
5805275,WI_2,HOUSEHOLD_2_406,11241,61.4600
5805276,WI_2,HOUSEHOLD_2_406,11242,61.4600
5805277,WI_2,HOUSEHOLD_2_406,11243,61.4600


Diễn giải: nếu key store-item-week là duy nhất và không có giá không dương, join giá sẽ không làm nổ dòng. Tuy vậy vẫn cần kiểm tra item-day có sales dương nhưng thiếu price sau khi nối.


## 7. Chuyển sales wide sang long theo cách kiểm soát bộ nhớ

Nếu melt toàn bộ bảng evaluation, số dòng long sẽ bằng số item-store series nhân với số ngày. Notebook không giữ toàn bộ bảng long 59 triệu dòng trong RAM cùng lúc; thay vào đó:

- tạo preview long để kiểm tra schema;
- tính phân phối units trực tiếp từ ma trận wide;
- khi join và aggregate, xử lý theo từng store và chỉ materialize các dòng `daily_units > 0` vì dòng zero không đóng góp revenue.

Cách này vẫn giữ được bảng aggregate đầy đủ bằng cách tạo grid date-store-dept/store sau đó fill doanh thu zero.

In [8]:
id_vars = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]


def melt_sales_chunk(sales_chunk, day_cols):
    return (
        sales_chunk.melt(
            id_vars=id_vars,
            value_vars=day_cols,
            var_name="d",
            value_name="daily_units",
        )
        .rename(columns={"daily_units": "daily_units"})
    )


long_sales_preview = melt_sales_chunk(selected_sales.head(3), selected_day_cols[:10])
sales_values = selected_sales[selected_day_cols].to_numpy(dtype=np.int16, copy=False)
long_total_rows = int(selected_sales.shape[0] * len(selected_day_cols))
unit_quantiles = np.quantile(sales_values, [0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0])
long_unit_summary = pd.DataFrame(
    [
        {"metric": "expected_long_rows", "value": long_total_rows},
        {"metric": "zero_unit_rows", "value": int((sales_values == 0).sum())},
        {"metric": "positive_unit_rows", "value": int((sales_values > 0).sum())},
        {"metric": "negative_unit_rows", "value": int((sales_values < 0).sum())},
        {"metric": "min_daily_units", "value": int(sales_values.min())},
        {"metric": "max_daily_units", "value": int(sales_values.max())},
        {"metric": "mean_daily_units", "value": float(sales_values.mean())},
    ]
)
unit_quantile_summary = pd.DataFrame(
    {
        "quantile": ["min", "25%", "50%", "75%", "90%", "95%", "99%", "max"],
        "daily_units": unit_quantiles,
    }
)

display(long_sales_preview)
display(long_unit_summary)
display(unit_quantile_summary)
print(
    "Diễn giải: bảng long có grain item-store-day. Số dòng zero lớn là kỳ vọng bình thường trong dữ liệu bán lẻ sparse; "
    "không nên xóa zero nếu mục tiêu là giữ đầy đủ chuỗi thời gian. Không có units âm trong kiểm tra này."
)

,id,item_id,dept_id,cat_id,store_id,state_id,d,daily_units
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_2,0
4,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_2,0
5,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_2,0
6,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_3,0
7,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_3,0
8,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_3,0
9,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_4,0


,metric,value
0,expected_long_rows,"59,181,090.0000"
1,zero_unit_rows,"40,241,819.0000"
2,positive_unit_rows,"18,939,271.0000"
3,negative_unit_rows,0.0000
4,min_daily_units,0.0000
5,max_daily_units,763.0000
6,mean_daily_units,1.1309


,quantile,daily_units
0,min,0.0000
1,25%,0.0000
2,50%,0.0000
3,75%,1.0000
4,90%,3.0000
5,95%,5.0000
6,99%,15.0000
7,max,763.0000


Diễn giải: bảng long có grain item-store-day. Số dòng zero lớn là kỳ vọng bình thường trong dữ liệu bán lẻ sparse; không nên xóa zero nếu mục tiêu là giữ đầy đủ chuỗi thời gian. Không có units âm trong kiểm tra này.


## 8. Join sales + calendar + prices và kiểm tra chất lượng

Logic join cần kiểm tra:

1. `sales long` nối `calendar` bằng `d`;
2. kết quả nối `sell_prices` bằng `store_id + item_id + wm_yr_wk`.

Để tránh giữ toàn bộ long table trong RAM, cell dưới đây xử lý từng store. Kiểm tra missing price toàn bộ item-day được tính bằng ma trận coverage giá; phần join item-level chỉ materialize các dòng có `daily_units > 0`.

In [9]:
state_to_snap_col = {"CA": "snap_CA", "TX": "snap_TX", "WI": "snap_WI"}
day_array = np.array(selected_day_cols, dtype=object)
weeks_by_day = calendar_join.set_index("d").loc[selected_day_cols, "wm_yr_wk"].to_numpy(dtype=np.int16)
calendar_d_duplicate_count = int(calendar_join.duplicated("d").sum())
calendar_missing_day_count = len(set(selected_day_cols) - set(calendar_join["d"].astype(str)))

positive_agg_chunks = []
join_quality_by_store_rows = []
total_missing_price_item_day_rows = 0
total_positive_units_missing_price_rows = 0
total_positive_rows_before_join = 0
total_positive_rows_after_calendar_join = 0
total_positive_rows_after_price_join = 0
negative_revenue_rows = 0
positive_units_zero_revenue_rows = 0

for store_id_value, sales_store in selected_sales.groupby("store_id", observed=True, sort=True):
    store_id = str(store_id_value)
    store_meta = sales_store[id_vars].reset_index(drop=True)
    store_values = sales_store[selected_day_cols].to_numpy(dtype=np.int16, copy=False)
    item_values = store_meta["item_id"].astype(str).to_numpy()
    dept_values = store_meta["dept_id"].astype(str).to_numpy()
    cat_values = store_meta["cat_id"].astype(str).to_numpy()
    state_values = store_meta["state_id"].astype(str).to_numpy()

    price_store = sell_prices[sell_prices["store_id"] == store_id]
    price_presence = (
        price_store.assign(has_price=True)
        .drop_duplicates(["item_id", "wm_yr_wk"])
        .pivot(index="item_id", columns="wm_yr_wk", values="has_price")
    )
    price_presence = (
        price_presence.reindex(index=item_values.tolist(), columns=weeks_by_day)
        .fillna(False)
    )
    has_price_matrix = price_presence.to_numpy(dtype=bool, copy=False)
    missing_price_matrix = ~has_price_matrix
    missing_price_rows = int(missing_price_matrix.sum())
    positive_missing_price_rows = int(((store_values > 0) & missing_price_matrix).sum())
    total_missing_price_item_day_rows += missing_price_rows
    total_positive_units_missing_price_rows += positive_missing_price_rows

    item_idx, day_idx = np.nonzero(store_values > 0)
    positive_sales = pd.DataFrame(
        {
            "item_id": item_values[item_idx],
            "dept_id": dept_values[item_idx],
            "cat_id": cat_values[item_idx],
            "store_id": store_id,
            "state_id": state_values[item_idx],
            "d": day_array[day_idx],
            "daily_units": store_values[item_idx, day_idx].astype("int16"),
        }
    )
    rows_before = len(positive_sales)
    total_positive_rows_before_join += rows_before

    positive_sales = positive_sales.merge(calendar_join, on="d", how="left", validate="many_to_one")
    rows_after_calendar = len(positive_sales)
    total_positive_rows_after_calendar_join += rows_after_calendar
    positive_missing_calendar = int(positive_sales["date"].isna().sum())

    price_store_for_merge = price_store.copy()
    price_store_for_merge["store_id"] = price_store_for_merge["store_id"].astype(str)
    price_store_for_merge["item_id"] = price_store_for_merge["item_id"].astype(str)
    positive_sales = positive_sales.merge(
        price_store_for_merge,
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left",
        validate="many_to_one",
    )
    rows_after_price = len(positive_sales)
    total_positive_rows_after_price_join += rows_after_price
    positive_join_missing_price = int(positive_sales["sell_price"].isna().sum())

    positive_sales["item_revenue"] = (
        positive_sales["daily_units"].astype("float32") * positive_sales["sell_price"].astype("float32")
    )
    negative_revenue_rows += int((positive_sales["item_revenue"] < 0).sum())
    positive_units_zero_revenue_rows += int(
        ((positive_sales["daily_units"] > 0) & (positive_sales["item_revenue"] == 0)).sum()
    )

    agg = (
        positive_sales.groupby(["date", "store_id", "dept_id"], observed=True)
        .agg(
            daily_units=("daily_units", "sum"),
            daily_revenue=("item_revenue", "sum"),
            active_item_count=("item_id", "nunique"),
        )
        .reset_index()
    )
    positive_agg_chunks.append(agg)

    join_quality_by_store_rows.append(
        {
            "store_id": store_id,
            "item_day_rows": int(store_values.size),
            "positive_unit_rows": rows_before,
            "missing_price_item_day_rows": missing_price_rows,
            "positive_units_missing_price_rows": positive_missing_price_rows,
            "positive_rows_after_calendar_join": rows_after_calendar,
            "positive_rows_after_price_join": rows_after_price,
            "positive_rows_missing_calendar": positive_missing_calendar,
            "positive_rows_missing_price_after_join": positive_join_missing_price,
        }
    )

    del price_presence, has_price_matrix, missing_price_matrix, positive_sales, agg, store_values
    gc.collect()

positive_store_dept_agg = pd.concat(positive_agg_chunks, ignore_index=True)
join_quality_by_store = pd.DataFrame(join_quality_by_store_rows)
join_quality_summary = pd.DataFrame(
    [
        {"check": "all_item_day_rows_before_join", "value": long_total_rows},
        {"check": "calendar_duplicate_d_count", "value": calendar_d_duplicate_count},
        {"check": "price_duplicate_store_item_week_key_count", "value": price_duplicate_keys},
        {"check": "calendar_missing_day_count", "value": calendar_missing_day_count},
        {"check": "all_item_day_rows_missing_price", "value": total_missing_price_item_day_rows},
        {"check": "positive_unit_rows_before_join", "value": total_positive_rows_before_join},
        {"check": "positive_rows_after_calendar_join", "value": total_positive_rows_after_calendar_join},
        {"check": "positive_rows_after_price_join", "value": total_positive_rows_after_price_join},
        {"check": "positive_unit_rows_missing_price", "value": total_positive_units_missing_price_rows},
        {"check": "negative_revenue_rows_after_reconstruction", "value": negative_revenue_rows},
        {"check": "positive_units_zero_revenue_rows", "value": positive_units_zero_revenue_rows},
    ]
)
join_quality_summary["rate_vs_all_item_day_rows"] = join_quality_summary["value"] / long_total_rows

display(join_quality_by_store)
display(join_quality_summary)
print(
    "Diễn giải: join không tạo nổ dòng nếu calendar d và price key đều duy nhất. "
    "Một số item-day thiếu price ở toàn bộ bảng long là vấn đề cần ghi nhận, nhưng điều quan trọng cho tái tạo doanh thu là "
    "các dòng có units dương có bị thiếu price hay không."
)

,store_id,item_day_rows,positive_unit_rows,missing_price_item_day_rows,positive_units_missing_price_rows,positive_rows_after_calendar_join,positive_rows_after_price_join,positive_rows_missing_calendar,positive_rows_missing_price_after_join
0,CA_1,5918109,2144779,1129842,0,2144779,2144779,0,0
1,CA_2,5918109,1845841,1556961,0,1845841,1845841,0,0
2,CA_3,5918109,2402766,1160796,0,2402766,2402766,0,0
3,CA_4,5918109,1657099,1265551,0,1657099,1657099,0,0
4,TX_1,5918109,1733173,1120154,0,1733173,1733173,0,0
5,TX_2,5918109,1993564,1110228,0,1993564,1993564,0,0
6,TX_3,5918109,1792295,1180942,0,1792295,1792295,0,0
7,WI_1,5918109,1849169,1357342,0,1849169,1849169,0,0
8,WI_2,5918109,1747023,1271529,0,1747023,1747023,0,0
9,WI_3,5918109,1773562,1146068,0,1773562,1773562,0,0


,check,value,rate_vs_all_item_day_rows
0,all_item_day_rows_before_join,59181090,1.0000
1,calendar_duplicate_d_count,0,0.0000
2,price_duplicate_store_item_week_key_count,0,0.0000
3,calendar_missing_day_count,0,0.0000
4,all_item_day_rows_missing_price,12299413,0.2078
5,positive_unit_rows_before_join,18939271,0.3200
6,positive_rows_after_calendar_join,18939271,0.3200
7,positive_rows_after_price_join,18939271,0.3200
8,positive_unit_rows_missing_price,0,0.0000
9,negative_revenue_rows_after_reconstruction,0,0.0000


Diễn giải: join không tạo nổ dòng nếu calendar d và price key đều duy nhất. Một số item-day thiếu price ở toàn bộ bảng long là vấn đề cần ghi nhận, nhưng điều quan trọng cho tái tạo doanh thu là các dòng có units dương có bị thiếu price hay không.


## 9. Tái tạo revenue ở mức item-day

M5 không cung cấp doanh thu trực tiếp. Sau khi kiểm tra join, revenue được tái tạo bằng:

```text
item_revenue = daily_units * sell_price
```

Đây là doanh thu ước tính từ giá tuần, không phải hóa đơn transaction-level. Vì vậy các kết luận về revenue cần nói rõ giới hạn này.

In [10]:
revenue_reconstruction_checks = pd.DataFrame(
    [
        {
            "check": "positive_unit_rows_missing_price",
            "value": total_positive_units_missing_price_rows,
            "meaning": "Nếu > 0 thì không thể tái tạo revenue đầy đủ cho các ngày có bán.",
        },
        {
            "check": "negative_revenue_rows",
            "value": negative_revenue_rows,
            "meaning": "Revenue âm là bất thường vì units và price đều nên không âm.",
        },
        {
            "check": "positive_units_zero_revenue_rows",
            "value": positive_units_zero_revenue_rows,
            "meaning": "Units dương nhưng revenue zero thường do price bằng 0 hoặc thiếu.",
        },
        {
            "check": "all_item_day_rows_missing_price",
            "value": total_missing_price_item_day_rows,
            "meaning": "Thiếu price ở toàn bộ item-day, thường liên quan giai đoạn item chưa có giá/không bán.",
        },
    ]
)
display(revenue_reconstruction_checks)
if total_positive_units_missing_price_rows == 0 and negative_revenue_rows == 0 and positive_units_zero_revenue_rows == 0:
    print(
        "Kết luận tạm thời: revenue có thể được tái tạo đủ tốt cho Phase tiếp theo ở các dòng có sales dương. "
        "Cần nhớ rằng giá là weekly price, không phải giá giao dịch từng ngày."
    )
else:
    print(
        "Kết luận tạm thời: revenue chưa an toàn để dùng trực tiếp; cần xử lý các dòng thiếu/không hợp lệ trước khi aggregate."
    )

,check,value,meaning
0,positive_unit_rows_missing_price,0,Nếu > 0 thì không thể tái tạo revenue đầy đủ c...
1,negative_revenue_rows,0,Revenue âm là bất thường vì units và price đều...
2,positive_units_zero_revenue_rows,0,Units dương nhưng revenue zero thường do price...
3,all_item_day_rows_missing_price,12299413,"Thiếu price ở toàn bộ item-day, thường liên qu..."


Kết luận tạm thời: revenue có thể được tái tạo đủ tốt cho Phase tiếp theo ở các dòng có sales dương. Cần nhớ rằng giá là weekly price, không phải giá giao dịch từng ngày.


## 10. Bảng ứng viên 1: `m5_store_dept_daily`

Grain: `date + store_id + dept_id`.

Bảng này phù hợp với scope `store + category/product group + date`. Các dòng không có sales dương vẫn được giữ bằng grid đầy đủ để doanh thu zero không biến mất khỏi chuỗi thời gian.

In [11]:
store_dept_meta = (
    selected_sales[["store_id", "state_id", "dept_id", "cat_id", "item_id"]]
    .assign(
        store_id=lambda df: df["store_id"].astype(str),
        state_id=lambda df: df["state_id"].astype(str),
        dept_id=lambda df: df["dept_id"].astype(str),
        cat_id=lambda df: df["cat_id"].astype(str),
        item_id=lambda df: df["item_id"].astype(str),
    )
    .groupby(["store_id", "state_id", "dept_id", "cat_id"], observed=True)
    .agg(item_count=("item_id", "nunique"))
    .reset_index()
)

date_features = calendar_join.copy()
full_store_dept_grid = date_features.merge(store_dept_meta, how="cross")

m5_store_dept_daily = full_store_dept_grid.merge(
    positive_store_dept_agg,
    on=["date", "store_id", "dept_id"],
    how="left",
)
for col in ["daily_units", "daily_revenue", "active_item_count"]:
    m5_store_dept_daily[col] = m5_store_dept_daily[col].fillna(0)

m5_store_dept_daily["daily_units"] = m5_store_dept_daily["daily_units"].astype("int32")
m5_store_dept_daily["active_item_count"] = m5_store_dept_daily["active_item_count"].astype("int16")
m5_store_dept_daily["daily_revenue"] = m5_store_dept_daily["daily_revenue"].astype("float32")
m5_store_dept_daily["weighted_avg_sell_price"] = np.where(
    m5_store_dept_daily["daily_units"] > 0,
    m5_store_dept_daily["daily_revenue"] / m5_store_dept_daily["daily_units"],
    np.nan,
)
m5_store_dept_daily["has_sales"] = (m5_store_dept_daily["daily_units"] > 0).astype("int8")
m5_store_dept_daily["snap_active"] = 0
for state, snap_col in state_to_snap_col.items():
    mask = m5_store_dept_daily["state_id"].eq(state)
    m5_store_dept_daily.loc[mask, "snap_active"] = m5_store_dept_daily.loc[mask, snap_col].astype("int8")

ordered_store_dept_cols = [
    "date",
    "d",
    "wm_yr_wk",
    "store_id",
    "state_id",
    "cat_id",
    "dept_id",
    "daily_revenue",
    "daily_units",
    "item_count",
    "active_item_count",
    "weighted_avg_sell_price",
    "has_sales",
    "weekday",
    "wday",
    "month",
    "year",
    "quarter",
    "week_of_year",
    "day_of_month",
    "day_of_year",
    "day_of_week",
    "day_of_week_num",
    "is_weekend",
    "year_month",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "event_count",
    "snap_CA",
    "snap_TX",
    "snap_WI",
    "snap_active",
]
m5_store_dept_daily = m5_store_dept_daily[ordered_store_dept_cols]

store_dept_checks = pd.DataFrame(
    [
        {"check": "shape_rows", "value": m5_store_dept_daily.shape[0]},
        {"check": "shape_columns", "value": m5_store_dept_daily.shape[1]},
        {
            "check": "duplicate_date_store_dept_keys",
            "value": int(m5_store_dept_daily.duplicated(["date", "store_id", "dept_id"]).sum()),
        },
        {"check": "zero_revenue_rows", "value": int((m5_store_dept_daily["daily_revenue"] == 0).sum())},
        {"check": "negative_revenue_rows", "value": int((m5_store_dept_daily["daily_revenue"] < 0).sum())},
        {"check": "total_revenue", "value": float(m5_store_dept_daily["daily_revenue"].sum())},
    ]
)
store_dept_missing = (
    m5_store_dept_daily.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).reset_index()
)
store_dept_missing.columns = ["column", "missing_count"]

display(m5_store_dept_daily.head())
display(store_dept_checks)
display(store_dept_missing)
display(
    m5_store_dept_daily[
        ["daily_revenue", "daily_units", "item_count", "active_item_count", "weighted_avg_sell_price"]
    ].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
)
print(
    "Diễn giải: một dòng của m5_store_dept_daily là doanh thu/units ước tính của một department tại một store trong một ngày. "
    "daily_units, active_item_count, has_sales và weighted_avg_sell_price là cột hữu ích cho EDA nhưng có nguy cơ leakage nếu dự báo doanh thu trước ngày bán."
)

,date,d,wm_yr_wk,store_id,state_id,cat_id,dept_id,daily_revenue,daily_units,item_count,active_item_count,weighted_avg_sell_price,has_sales,weekday,wday,month,year,quarter,week_of_year,day_of_month,day_of_year,day_of_week,day_of_week_num,is_weekend,year_month,event_name_1,event_type_1,event_name_2,event_type_2,event_count,snap_CA,snap_TX,snap_WI,snap_active
0,2011-01-29,d_1,11101,CA_1,CA,FOODS,FOODS_1,681.1800,297,216,70,2.2935,1,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0
1,2011-01-29,d_1,11101,CA_1,CA,FOODS,FOODS_2,"2,236.0100",674,398,154,3.3175,1,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0
2,2011-01-29,d_1,11101,CA_1,CA,FOODS,FOODS_3,"4,323.4600",2268,823,285,1.9063,1,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0
3,2011-01-29,d_1,11101,CA_1,CA,HOBBIES,HOBBIES_1,"1,276.8600",528,416,101,2.4183,1,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0
4,2011-01-29,d_1,11101,CA_1,CA,HOBBIES,HOBBIES_2,93.0500,28,149,17,3.3232,1,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0


,check,value
0,shape_rows,"135,870.0000"
1,shape_columns,34.0000
2,duplicate_date_store_dept_keys,0.0000
3,zero_revenue_rows,363.0000
4,negative_revenue_rows,0.0000
5,total_revenue,"191,577,536.0000"


,column,missing_count
0,event_name_2,135590
1,event_type_2,135590
2,event_name_1,124810
3,event_type_1,124810
4,weighted_avg_sell_price,363


,daily_revenue,daily_units,item_count,active_item_count,weighted_avg_sell_price
count,"135,870.0000","135,870.0000","135,870.0000","135,870.0000","135,507.0000"
mean,"1,410.0061",492.5824,435.5714,139.3926,3.2682
std,"1,384.2343",603.8029,206.2730,107.7646,1.0581
min,0.0000,0.0000,149.0000,0.0000,0.2000
1%,15.3800,6.0000,149.0000,5.0000,1.4988
5%,42.6345,18.0000,149.0000,12.0000,1.9266
50%,985.7900,288.0000,416.0000,111.0000,3.1447
95%,"4,258.8291","1,841.5500",823.0000,370.0000,5.1735
99%,"6,484.8793","2,878.3100",823.0000,471.0000,5.7246
max,"11,198.9502","5,118.0000",823.0000,621.0000,10.9650


Diễn giải: một dòng của m5_store_dept_daily là doanh thu/units ước tính của một department tại một store trong một ngày. daily_units, active_item_count, has_sales và weighted_avg_sell_price là cột hữu ích cho EDA nhưng có nguy cơ leakage nếu dự báo doanh thu trước ngày bán.


## 11. Bảng ứng viên 2: `m5_store_daily`

Grain: `date + store_id`.

Bảng này được tổng hợp từ `m5_store_dept_daily` để giữ nhất quán calendar, state và cách fill zero. Đổi lại, nó mất chi tiết category/department.

In [12]:
m5_store_daily = (
    m5_store_dept_daily.groupby(["date", "d", "wm_yr_wk", "store_id", "state_id"], observed=True)
    .agg(
        daily_revenue=("daily_revenue", "sum"),
        daily_units=("daily_units", "sum"),
        item_count=("item_count", "sum"),
        active_item_count=("active_item_count", "sum"),
        dept_count=("dept_id", "nunique"),
        active_dept_count=("daily_units", lambda s: int((s > 0).sum())),
        weekday=("weekday", "first"),
        wday=("wday", "first"),
        month=("month", "first"),
        year=("year", "first"),
        quarter=("quarter", "first"),
        week_of_year=("week_of_year", "first"),
        day_of_month=("day_of_month", "first"),
        day_of_year=("day_of_year", "first"),
        day_of_week=("day_of_week", "first"),
        day_of_week_num=("day_of_week_num", "first"),
        is_weekend=("is_weekend", "first"),
        year_month=("year_month", "first"),
        event_name_1=("event_name_1", "first"),
        event_type_1=("event_type_1", "first"),
        event_name_2=("event_name_2", "first"),
        event_type_2=("event_type_2", "first"),
        event_count=("event_count", "first"),
        snap_CA=("snap_CA", "first"),
        snap_TX=("snap_TX", "first"),
        snap_WI=("snap_WI", "first"),
        snap_active=("snap_active", "first"),
    )
    .reset_index()
)
m5_store_daily["weighted_avg_sell_price"] = np.where(
    m5_store_daily["daily_units"] > 0,
    m5_store_daily["daily_revenue"] / m5_store_daily["daily_units"],
    np.nan,
)
m5_store_daily["has_sales"] = (m5_store_daily["daily_units"] > 0).astype("int8")

store_daily_checks = pd.DataFrame(
    [
        {"check": "shape_rows", "value": m5_store_daily.shape[0]},
        {"check": "shape_columns", "value": m5_store_daily.shape[1]},
        {
            "check": "duplicate_date_store_keys",
            "value": int(m5_store_daily.duplicated(["date", "store_id"]).sum()),
        },
        {"check": "zero_revenue_rows", "value": int((m5_store_daily["daily_revenue"] == 0).sum())},
        {"check": "negative_revenue_rows", "value": int((m5_store_daily["daily_revenue"] < 0).sum())},
        {"check": "total_revenue", "value": float(m5_store_daily["daily_revenue"].sum())},
    ]
)
store_daily_missing = (
    m5_store_daily.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).reset_index()
)
store_daily_missing.columns = ["column", "missing_count"]

display(m5_store_daily.head())
display(store_daily_checks)
display(store_daily_missing)
display(
    m5_store_daily[
        [
            "daily_revenue",
            "daily_units",
            "item_count",
            "active_item_count",
            "dept_count",
            "active_dept_count",
            "weighted_avg_sell_price",
        ]
    ].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
)
print(
    "Diễn giải: một dòng của m5_store_daily là tổng doanh thu/units của một store trong một ngày. "
    "Bảng này đơn giản hơn cho dự báo cấp store nhưng không còn trả lời được câu hỏi nhóm sản phẩm nào đóng góp doanh thu."
)

,date,d,wm_yr_wk,store_id,state_id,daily_revenue,daily_units,item_count,active_item_count,dept_count,active_dept_count,weekday,wday,month,year,quarter,week_of_year,day_of_month,day_of_year,day_of_week,day_of_week_num,is_weekend,year_month,event_name_1,event_type_1,event_name_2,event_type_2,event_count,snap_CA,snap_TX,snap_WI,snap_active,weighted_avg_sell_price,has_sales
0,2011-01-29,d_1,11101,CA_1,CA,"10,933.1602",4337,3049,837,7,7,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,2.5209,1
1,2011-01-29,d_1,11101,CA_2,CA,"9,101.5195",3494,3049,724,7,7,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,2.6049,1
2,2011-01-29,d_1,11101,CA_3,CA,"11,679.8301",4739,3049,872,7,7,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,2.4646,1
3,2011-01-29,d_1,11101,CA_4,CA,"4,561.5898",1625,3049,588,7,6,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,2.8071,1
4,2011-01-29,d_1,11101,TX_1,TX,"6,586.6797",2556,3049,662,7,7,Saturday,1,1,2011,1,4,29,29,Saturday,5,1,2011-01,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,2.5769,1


,check,value
0,shape_rows,"19,410.0000"
1,shape_columns,34.0000
2,duplicate_date_store_keys,0.0000
3,zero_revenue_rows,24.0000
4,negative_revenue_rows,0.0000
5,total_revenue,"191,577,536.0000"


,column,missing_count
0,event_name_2,19370
1,event_type_2,19370
2,event_name_1,17830
3,event_type_1,17830
4,weighted_avg_sell_price,24


,daily_revenue,daily_units,item_count,active_item_count,dept_count,active_dept_count,weighted_avg_sell_price
count,"19,410.0000","19,410.0000","19,410.0000","19,410.0000","19,410.0000","19,410.0000","19,386.0000"
mean,"9,870.0430","3,448.0769","3,049.0000",975.7481,7.0000,6.9813,2.8534
std,"3,944.3469","1,317.4214",0.0000,256.7954,0.0000,0.3321,0.2426
min,0.0000,0.0000,"3,049.0000",0.0000,7.0000,0.0000,0.8200
1%,"3,574.8142","1,306.0000","3,049.0000",449.0000,7.0000,7.0000,2.3752
5%,"4,582.9756","1,691.0000","3,049.0000",569.4500,7.0000,7.0000,2.5012
50%,"9,172.7300","3,240.0000","3,049.0000",963.0000,7.0000,7.0000,2.8256
95%,"17,659.9194","5,999.0000","3,049.0000","1,432.0000",7.0000,7.0000,3.2877
99%,"22,051.0236","7,497.0000","3,049.0000","1,613.9100",7.0000,7.0000,3.4290
max,"29,381.8809","9,338.0000","3,049.0000","1,940.0000",7.0000,7.0000,3.7267


Diễn giải: một dòng của m5_store_daily là tổng doanh thu/units của một store trong một ngày. Bảng này đơn giản hơn cho dự báo cấp store nhưng không còn trả lời được câu hỏi nhóm sản phẩm nào đóng góp doanh thu.


## 12. So sánh hai grain ứng viên

So sánh này dùng các bảng vừa tạo, không kết luận trước. Mục tiêu là chọn grain phù hợp hơn cho Phase tiếp theo của project retail revenue forecasting.

In [13]:
comparison = pd.DataFrame(
    [
        {
            "table": "m5_store_dept_daily",
            "row_count": m5_store_dept_daily.shape[0],
            "row_meaning": "date + store_id + dept_id",
            "product_detail": "giữ department và category",
            "target_meaning": "doanh thu ngày của một department trong một store",
            "zero_revenue_rows": int((m5_store_dept_daily["daily_revenue"] == 0).sum()),
            "zero_revenue_rate": float((m5_store_dept_daily["daily_revenue"] == 0).mean()),
            "modeling_complexity": "cao hơn nhưng vẫn gọn sau aggregate",
            "compatibility_with_maven": "phù hợp hơn nếu chuẩn hóa Maven về category/product group",
            "recommended_next_use": "khuyến nghị cho phase tiếp theo",
        },
        {
            "table": "m5_store_daily",
            "row_count": m5_store_daily.shape[0],
            "row_meaning": "date + store_id",
            "product_detail": "mất chi tiết sản phẩm",
            "target_meaning": "tổng doanh thu ngày của một store",
            "zero_revenue_rows": int((m5_store_daily["daily_revenue"] == 0).sum()),
            "zero_revenue_rate": float((m5_store_daily["daily_revenue"] == 0).mean()),
            "modeling_complexity": "thấp hơn",
            "compatibility_with_maven": "dùng tốt cho baseline store-level nhưng khó so sánh nhóm sản phẩm",
            "recommended_next_use": "dùng làm baseline phụ hoặc sanity check",
        },
    ]
)
display(comparison)
print(
    "Khuyến nghị: m5_store_dept_daily phù hợp hơn cho phase tiếp theo vì giữ được nhóm sản phẩm, "
    "vẫn có kích thước vừa phải, và khớp với scope store + category/product group + date. "
    "m5_store_daily nên dùng như bảng baseline cấp store."
)

,table,row_count,row_meaning,product_detail,target_meaning,zero_revenue_rows,zero_revenue_rate,modeling_complexity,compatibility_with_maven,recommended_next_use
0,m5_store_dept_daily,135870,date + store_id + dept_id,giữ department và category,doanh thu ngày của một department trong một store,363,0.0027,cao hơn nhưng vẫn gọn sau aggregate,phù hợp hơn nếu chuẩn hóa Maven về category/pr...,khuyến nghị cho phase tiếp theo
1,m5_store_daily,19410,date + store_id,mất chi tiết sản phẩm,tổng doanh thu ngày của một store,24,0.0012,thấp hơn,dùng tốt cho baseline store-level nhưng khó so...,dùng làm baseline phụ hoặc sanity check


Khuyến nghị: m5_store_dept_daily phù hợp hơn cho phase tiếp theo vì giữ được nhóm sản phẩm, vẫn có kích thước vừa phải, và khớp với scope store + category/product group + date. m5_store_daily nên dùng như bảng baseline cấp store.


## 13. Kết luận Phase 1 + 2

Kết luận dưới đây chỉ dựa trên các output đã chạy trong notebook:

- Đọc dữ liệu gốc thành công và thư mục M5 có đủ 5 file chính.
- `sales_train_evaluation.csv` được chọn vì có cùng metadata item/store với validation nhưng bao phủ nhiều ngày hơn.
- Grain tự nhiên của sales gốc là item-store series dạng wide; sau khi long hóa là item-store-day.
- Calendar nối với sales bằng `d`; price nối bằng `store_id + item_id + wm_yr_wk`.
- Price là weekly store-item price, nên revenue tái tạo là ước tính từ `daily_units * sell_price`, không phải invoice revenue.
- Join không bị nổ dòng khi key calendar và price đều duy nhất.
- Có nhiều item-day thiếu price trong toàn bộ grid, nhưng các dòng có units dương không bị thiếu price trong kiểm tra này; do đó revenue aggregate đủ tốt cho phase tiếp theo.
- `m5_store_dept_daily` nên là bảng chính cho phase tiếp theo; `m5_store_daily` là bảng phụ để baseline hoặc so sánh cấp store.
- Cần cẩn thận leakage: `daily_units`, `active_item_count`, `has_sales`, `active_dept_count`, và `weighted_avg_sell_price` đều dùng thông tin cùng ngày, không an toàn nếu mục tiêu là dự báo trước ngày bán.
- Việc tiếp theo nên là EDA sâu hơn trên `m5_store_dept_daily`, chuẩn hóa schema với Maven Market, và chỉ sau đó mới xây baseline forecasting theo thời gian.